In [ ]:
FOLDX_PARAMS = {  # FoldX defaults — script sets none explicitly
    'numberOfRuns': 1,
    'pH':           7,
    'temperature':  298,
    'ionStrength':  0.05,
}

In [ ]:
NUMBERING_CONFIRMED = True
TM_KEYS    = ['position', 'wildtype', 'mutation']   # merge on these, not a string
TM_DDG_COL = 'ddG_pred'
TM_OFFSET  = 29     # ThermoMPNN position = PDB - 29

# ThermoMPNN: NEGATIVE ddG_pred = stabilizing (Kuhlman Lab; confirmed on our 5300-row distribution)
THERMOMPNN_STABILIZING = lambda d: d < 0
FOLDX_STABILIZING      = lambda d: d < 0     # FoldX 'total energy', negative = stabilizing

## Diagnostic Test 1 — interpretation (Day 15)

**Finding: Scenario A (bug).** filter_candidates_v2.py had ThermoMPNN's sign
inverted. ThermoMPNN reports negative ddG_pred as stabilizing, but the filter
selected `ddG_pred > 0.5` and sorted descending — selecting the 30 *most
destabilizing* mutations and labeling them stabilizing. FoldX then correctly
rejected all 30. The tools never disagreed; the filter turned agreement into
apparent conflict.

**Evidence (our data):** of 5300 predictions, 4517 positive / 518 negative,
mean +1.1, range -1.86 to +4.21. A folded protein is mostly destabilized by
mutation, so negative=stabilizing is the only consistent convention.

**Caveat — known-stabilizer recovery is partial.** As singles, only N233K
(-1.20) and D186H (-0.51) read clearly stabilizing; S238F, S121E, R280A are
near-neutral; R224Q (+0.38) and W159H (+1.25) read destabilizing. Expected:
these were combination-validated, single effects are smaller, epistasis can
flip signs. ThermoMPNN's stabilizing calls are usable but noisy.

**Implication:** the bug — not tool disagreement — explains Phase 1 v1.
Fix: select `ddG_pred < -0.5`, sort ascending. Because ThermoMPNN's
stabilizing predictions are noisy even on validated mutations, the corrected
candidate set must still be FoldX-cross-validated (consensus, not rank alone).